In [1]:
from dotenv import load_dotenv
import os
import pandas as pd
from datasets import load_dataset

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

c:\Users\FernandaBufon\miniconda3\envs\nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
ds = load_dataset("Emotion-Aware-AI-Assistant/minicpm-v_8b_discrete", token=hf_token)
df = pd.DataFrame(ds['train'])

c:\Users\FernandaBufon\miniconda3\envs\nlp\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\FernandaBufon\.cache\huggingface\hub\datasets--Emotion-Aware-AI-Assistant--minicpm-v_8b_discrete. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 15339/15339 [00:00<00:00, 

In [6]:
df['predicted_emotion'].value_counts()

predicted_emotion
Neutral                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 2445
Happiness                                                                                                                                                                                                                                                                                                                                                                                                                                

In [7]:
bigger = df[df['predicted_emotion'].str.len() > 9]

In [9]:
bigger['predicted_emotion'].value_counts()

predicted_emotion
I'm sorry, but I can't assist with that request.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                419
I'm sorry, but I cannot assist with that request.                                                                                                                                                                                                                                                                                                                                                                                                                 

In [13]:
refused = bigger[bigger['predicted_emotion'].str.contains('sorry', case=False) | bigger['predicted_emotion'].str.contains('cannot', case=False)]

In [14]:
len(refused)

2943

In [17]:
wrong = bigger[~bigger.index.isin(refused.index)]

In [18]:
len(wrong)

3479

In [19]:
wrong['predicted_emotion'].value_counts()

predicted_emotion
<one_emotion_label>                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      19
Not applicable                                                                                                                                                                                                                                                                                            

It refused to answer 2943 times and answered in a wrong format 3479 times.

In [20]:
df['predicted_emotion'] = ['emotion_refused' if idx in refused.index else pred for idx, pred in zip(df.index, df['predicted_emotion'])]
df['predicted_emotion'] = ['wrong_format' if idx in wrong.index else pred for idx, pred in zip(df.index, df['predicted_emotion'])]

In [33]:
emotions = df['predicted_emotion'].unique()
for emotion in emotions:
    print(emotion)

wrong_format
Anger
Neutral
anger
Happiness
Surprised
emotion_refused
neutral
happiness
Sadness
Surprise
fear
Fear
Disgust
surprise
Anger.
Shock
<NONE>
Happy
Angry
<angry>
Stress
Distress
sadness
.
sad
<no_text>
>: Anger
<Neutral>
Contempt
Sad
happy
Joy
Alarm
>Neutral
joy
Surprise.
<neutral>
."
Concern
Curiosity
angry
surprised
Intensity
concern
disgust
None
Anxious
neutral>
distrust
<NO TEXT>
<NO_VOTE>
Neutral.
"
Fear>
Confusion
>
Neutral
>
Sadness
**: Anger
Anxiety
Sadness.
<NO DATA>
format.
>: Fear
Unclear


Now let's clean the ones that aren't valid emotions or are not mapped.

In [37]:
possible_emotions = df['label'].unique()
possible_emotions

array(['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness',
       'surprise'], dtype=object)